# Context-sensitive Spelling Correction

The goal of the assignment is to implement context-sensitive spelling correction. The input of the code will be a set of text lines and the output will be the same lines with spelling mistakes fixed.

Submit the solution of the assignment to Moodle as a link to your GitHub repository containing this notebook.

Useful links:
- [Norvig's solution](https://norvig.com/spell-correct.html)
- [Norvig's dataset](https://norvig.com/big.txt)
- [Ngrams data](https://www.ngrams.info/download_coca.asp)

Grading:
- 60 points - Implement spelling correction
- 20 points - Justify your decisions
- 20 points - Evaluate on a test set


## Implement context-sensitive spelling correction

Your task is to implement context-sensitive spelling corrector using N-gram language model. The idea is to compute conditional probabilities of possible correction options. For example, the phrase "dking sport" should be fixed as "doing sport" not "dying sport", while "dking species" -- as "dying species".

The best way to start is to analyze [Norvig's solution](https://norvig.com/spell-correct.html) and [N-gram Language Models](https://web.stanford.edu/~jurafsky/slp3/3.pdf).

When solving this task, we expect you'll face (and successfully deal with) some problems or make up the ideas of the model improvement. Some of them are: 

- solving a problem of n-grams frequencies storing for a large corpus;
- taking into account keyboard layout and associated misspellings;
- efficiency improvement to make the solution faster;
- ...

Please don't forget to describe such cases, and what you decided to do with them, in the Justification section.

##### IMPORTANT:  
Your project should not be a mere code copy-paste from somewhere. You must provide:
- Your implementation
- Analysis of why the implemented approach is suggested
- Improvements of the original approach that you have chosen to implement

In [253]:
!pip install python-Levenshtein

Defaulting to user installation because normal site-packages is not writeable
  Using cached levenshtein-0.27.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.6 kB)
Using cached levenshtein-0.27.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (159 kB)

[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [389]:
import heapq
import numpy as np
import re
import Levenshtein

In [390]:
TOP_K = 150  # Number of candidates

In [391]:
VOCAB = {}

In [392]:
with open("data/big.txt", "r") as f:
	words = re.findall(r'\w+', f.read().lower())

prev = None
for word in words:
	entry = VOCAB.setdefault(word, [0, {}])
	entry[0] += 1

	if prev is not None:
		entry[1][prev] = entry[1].get(prev, 0) + 1
	prev = word

In [393]:
data = []
with open("data/bigrams.txt", "rb") as f:
	for byte_line in f:
		try:
			line = byte_line.decode("utf-8")
			data.append(line.strip())
		except UnicodeDecodeError:
			pass

for line in data:
	items = line.split()
	occur, w1, w2 = int(items[0]), items[1].lower(), items[2].lower()

	entry = VOCAB.setdefault(w2, [0, {}])
	entry[0] += 1

	entry[1][w1] = entry[1].get(w1, 0) + 1

In [394]:
def find_candidates(word, n=TOP_K):
	# Find candidates
	cands = heapq.nsmallest(n, VOCAB.keys(), key=lambda x: Levenshtein.distance(word, x))
	return cands


def calc_prob(w1, w2, f=False):
	# Calculate the Likelihood
	if w1 not in VOCAB.keys() or w2 not in VOCAB.keys():
		return 0

	w1_stats = VOCAB.get(w1)
	w2_stats = VOCAB.get(w2)

	if w1_stats[0] == 0 or w2_stats[0] == 0:
		return 0

	s = w2_stats[1].get(w1)
	if s is None:
		s = 0

	p_ba = s / w1_stats[0]
	p_a = w1_stats[0] / len(VOCAB.keys())
	p_b = w2_stats[0] / len(VOCAB.keys())

	a = p_ba * p_a / p_b
	if f:
		a = p_ba * p_a 

	return a


def Prob(c, prev, next):
	# Calculate probability looking at neighbours
	prev_prob = calc_prob(prev, c)
	next_prob = calc_prob(c, next)
	return prev_prob + next_prob


def correct_word(word, prev, next, beam, n=TOP_K):
	# Find candidates
	cands = find_candidates(word, n)
	top_n = heapq.nlargest(beam, cands, key=lambda x: Prob(x, prev, next))
	return top_n


def seq_likelihood(words):
	score = 0.0
	for i in range(1, len(words)):
		score += calc_prob(words[i - 1], words[i], True)
	return score / len(words[:-1])


def beam_search(results, init_words):
	best_score = seq_likelihood(init_words)
	ans = init_words

	for candidates, idx in results:
		for w in candidates:
			seq = init_words[:idx] + [w] + init_words[idx + 1:]
			score = seq_likelihood(seq)
			
			if score > best_score:
				best_score = score
				ans = seq
			
	return ans, best_score


def make_correction(text, beam=5):
	words = text.lower().split()
	results = []

	for i in range(len(words)):
		prev = words[i - 1] if i > 0 else None
		next = words[i + 1] if i < len(words) - 1 else None

		res = correct_word(words[i], prev, next, beam)
		results.append([res, i])
	
	ans, best_score = beam_search(results, words)
	# print(best_score)
	return " ".join(ans)

In [395]:
beam_size = 150
text = "what is wrpng"
make_correction(text, beam_size)

'that is wrpng'

## Justify your decisions

Write down justificaitons for your implementation choices. For example, these choices could be:
- Which ngram dataset to use
- Which weights to assign for edit1, edit2 or absent words probabilities
- Beam search parameters
- etc.

## Homework

This solution rely on statistics of bi-grams occurance. No BERT, ML or other. Only statistics.

For reference text I used the same 'big.txt' file as Norvig.
Difference with norvig solution lies in filtering candidates and calculating probability for each.

To choose candidates I used levenshtein distance, as it considers all possible changes of the word.
Unlike Norvig solution that considers only candidates with levenstein distance 2.

The probability for each candidate is calculated according to neighbour words. Vocabulary contains the list of previous words and its frequency. Hence, we can apply statistics to know the chance of each candidate to be here.

At the end for each word I leave 3 candidates, to evaluate the final sequence. Each candidate put inside seqeunce to evaluate its probability. Sequence with highest probability is considered as fixed result. I took the inspiration of the beam search for this last check.

Results not perfect and algorithm still fix only one word in whole sequence.

## Evaluate on a test set

Yx`our task is to generate a test set and evaluate your work. You may vary the noise probability to generate different datasets with varying compexity (or just take another dataset). Compare your solution to the Norvig's corrector, and report the accuracies.

In [396]:
import re
from collections import Counter

def words(text): return re.findall(r'\w+', text.lower())

WORDS = Counter(words(open('data/big.txt').read()))

def P(word, N=sum(WORDS.values())): 
    "Probability of `word`."
    return WORDS[word] / N

def correction(word): 
    "Most probable spelling correction for word."
    return max(candidates(word), key=P)

def candidates(word): 
    "Generate possible spelling corrections for word."
    return (known([word]) or known(edits1(word)) or known(edits2(word)) or [word])

def known(words): 
    "The subset of `words` that appear in the dictionary of WORDS."
    return set(w for w in words if w in WORDS)

def edits1(word):
    "All edits that are one edit away from `word`."
    letters    = 'abcdefghijklmnopqrstuvwxyz'
    splits     = [(word[:i], word[i:])    for i in range(len(word) + 1)]
    deletes    = [L + R[1:]               for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R)>1]
    replaces   = [L + c + R[1:]           for L, R in splits if R for c in letters]
    inserts    = [L + c + R               for L, R in splits for c in letters]
    return set(deletes + transposes + replaces + inserts)

def edits2(word): 
    "All edits that are two edits away from `word`."
    return (e2 for e1 in edits1(word) for e2 in edits1(e1))

In [397]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize, TreebankWordDetokenizer
import random
from tqdm import tqdm

nltk.download('punkt')

[nltk_data] Downloading package punkt to /home/t/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [398]:
def create_typo(word):
    if len(word) == 0:
        return word
    original = word
    for _ in range(10):
        op = random.choice(['insert', 'delete', 'replace', 'swap'])
        pos = random.randint(0, len(word)-1)
        new_word = original
        
        if op == 'insert':
            char = random.choice('abcdefghijklmnopqrstuvwxyz')
            new_word = word[:pos] + char + word[pos:]
        elif op == 'delete':
            new_word = word[:pos] + word[pos+1:]
        elif op == 'replace':
            char = random.choice('abcdefghijklmnopqrstuvwxyz')
            new_word = word[:pos] + char + word[pos+1:]
        elif op == 'swap' and len(word) > 1:
            pos = min(pos, len(word)-2)
            new_word = word[:pos] + word[pos+1] + word[pos] + word[pos+2:]
        
        if new_word != original:
            return new_word
    return original

def generate_dataset(file_path='data/big.txt', num_samples=1000):
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()
    
    sentences = sent_tokenize(text)
    detokenizer = TreebankWordDetokenizer()
    dataset = []
    
    for sent in sentences[:100]:
        tokens = word_tokenize(sent)
        tokens = tokens[:12]
        alpha_indices = [i for i, t in enumerate(tokens) if t.isalpha()]
        
        if not alpha_indices or len(alpha_indices) < 3:
            continue
        
        target_idx = random.choice(alpha_indices)
        original_word = tokens[target_idx]
        typo_word = create_typo(original_word)
        
        if typo_word == original_word:
            continue
        
        corrupted_tokens = tokens.copy()
        corrupted_tokens[target_idx] = typo_word
        original_sent = detokenizer.detokenize(tokens)
        corrupted_sent = detokenizer.detokenize(corrupted_tokens)
        
        dataset.append((corrupted_sent, original_sent))
        
        if len(dataset) >= num_samples:
            break
    
    return dataset

In [399]:
def evaluate_norvig(dataset):
    detokenizer = TreebankWordDetokenizer()
    correct = 0
    total = len(dataset)
    
    for corrupted, original in tqdm(dataset):
        tokens = word_tokenize(corrupted)
        corrected_tokens = []
        for token in tokens:
            if token.isalpha():
                corrected = correction(token.lower()).lower()
                corrected_tokens.append(corrected)
            else:
                corrected_tokens.append(token)
        corrected_sent = detokenizer.detokenize(corrected_tokens)
        if corrected_sent.lower() == original.lower():
            correct += 1
    
    return correct / total

def evaluate_my_solution(dataset, my_spell_checker):
    correct = 0
    total = len(dataset)
    for corrupted, original in tqdm(dataset):
        corrected = my_spell_checker(corrupted)
        if corrected.lower() == original.lower():
            correct += 1
    return correct / total

In [400]:
# Generate dataset
dataset = generate_dataset(file_path='data/big.txt', num_samples=1000)

# Evaluate
norvig_accuracy = evaluate_norvig(dataset)
my_accuracy = evaluate_my_solution(dataset, make_correction)

print(f"Norvig's Accuracy: {norvig_accuracy:.2f}")
print(f"My Solution's Accuracy: {my_accuracy:.2f}")

100%|██████████| 94/94 [00:21<00:00,  4.39it/s]

Norvig's Accuracy: 0.65
My Solution's Accuracy: 0.16


#### Useful resources (also included in the archive in moodle):

1. [Possible dataset with N-grams](https://www.ngrams.info/download_coca.asp)
2. [Damerau–Levenshtein distance](https://en.wikipedia.org/wiki/Damerau–Levenshtein_distance#:~:text=Informally%2C%20the%20Damerau–Levenshtein%20distance,one%20word%20into%20the%20other.)